# Parameter Calibration from LOBSTER Data

Companion notebook to the appendix *Parameter Calibration* of the thesis chapter *Trading with the Flow* (the two-sided resilience model of the limit order book and its volume-imbalance extension). It documents the choice of

* the order-size distributions $P^1$, $P^2$ and the scale $Q$ between model liquidity units and shares (Section 3),
* the resilience parameters $(\kappa, \theta_f, \theta_g)$ and the liquidity capacity $\lambda_{\max}$ by Jensen--Shannon calibration of the stationary liquidity law (Sections 2, 4, 5),
* the volume-imbalance sensitivity $\hat{\kappa}$ by conditional maximum likelihood (Section 6).

The empirical input is LOBSTER level-2 message and order-book data for 2025-07-16, 10:30--15:00, for the tickers AAPL, AMAT, CSCO, EBAY, GOOG, INTC and MSFT. LOBSTER data is licensed and **not** part of this repository; place the files `lobster_data/{TICKER}_2025-07-16_34140000_57660000_{message,orderbook}_2.csv` next to this notebook to re-run it. All cells that need the data are marked as such. Outputs have been cleared.

In [ ]:
import copy
import math
import time
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import tqdm
from numpy.linalg import solve
from scipy.optimize import minimize, minimize_scalar, brentq

## 1. Model Specification

For activity parameters $\theta_f, \theta_g \ge 0$, resilience parameter $\kappa > 0$, and reference liquidity level $\lambda_r \ge 0$, we define the rate functions
$$f(\lambda) := \theta_f \, e^{-\kappa(\lambda - \lambda_r)}, \qquad g(\lambda) := \theta_g \, e^{\kappa(\lambda - \lambda_r)},$$
governing limit-order and market-order arrivals, respectively. The reference liquidity level $\lambda_r$ is chosen as the stationary mean $\bar{\lambda}_\pi$ of the liquidity process for fixed $(\kappa, \theta_f, \theta_g)$ and the finite liquidity state space $\Lambda = \{0, 1, \ldots, \lambda_{\max}\}$.

In [ ]:
# =====================================================================
# Liquidity state space (model units)
# =====================================================================
liq_min = 0
liq_max = 20
liq_grid = np.arange(liq_min, liq_max + 1, 1.0)

# =====================================================================
# Mark space: order-size distributions, thesis Subsection "Model Specifications"
#   P1: market-order sizes (sign = direction: +buy hits the ask, -sell hits the bid)
#   P2: limit-order sizes  (sign: +posting, -cancellation)
#   P3: order type          (-1: market order, 1: bid limit order, 2: ask limit order)
# =====================================================================
P1_vals = {-3: 0.075, -2: 0.125, -1: 0.3, 1: 0.3, 2: 0.125, 3: 0.075}

# Limit-order size distribution of Bank, Cartea, Koerber (BCK): more weight on
# large postings, produces a negatively skewed stationary law -- kept only for
# the comparison in Section 2.
P2_vals_old = {-3: 0.1, -2: 0.075, -1: 0.075, 1: 0.25, 2: 0.25, 3: 0.25}

# Our limit-order size distribution, motivated by the empirical categories of
# Table 1b (core-group average 0.006, 0.115, 0.341, 0.390, 0.139, 0.009).
P2_vals = {-3: 0.01, -2: 0.1, -1: 0.275, 1: 0.35, 2: 0.25, 3: 0.015}

P3_vals = {-1: 0.5, 1: 0.25, 2: 0.25}


def build_E_marks(P1_vals, P2_vals, P3_vals):
    """Product measure nu = P1 x P2 x P3 as a list of marks with their weights."""
    E_marks: List[Dict[str, Any]] = []
    for e1, p1 in P1_vals.items():
        for e2, p2 in P2_vals.items():
            for e3, p3 in P3_vals.items():
                nu = float(p1 * p2 * p3)
                if e3 == -1:
                    E_marks.append({'kind': 'market', 'eta': float(e1), 'rho': None, 'nu': float(nu)})
                elif e3 == 1:
                    E_marks.append({'kind': 'limit_buy', 'eta': None, 'rho': float(e2), 'nu': float(nu)})
                elif e3 == 2:
                    E_marks.append({'kind': 'limit_sell', 'eta': None, 'rho': float(e2), 'nu': float(nu)})
                else:
                    raise ValueError('Unexpected e3')
    return E_marks


E_marks = build_E_marks(P1_vals, P2_vals, P3_vals)

# Initial resilience parameters (replaced by the calibration below)
theta_f = 100.0
theta_g = 25.0
kappa = 0.1
lam_ref = 10.0

In [ ]:
def f(liquidity: float, theta_f: float, kappa: float, lam_ref: float) -> float:
    """Arrival intensity for limit orders."""
    return theta_f * np.exp(-kappa * (float(max(liquidity, 0.0)) - lam_ref))

def g(liquidity: float, theta_g: float, kappa: float, lam_ref: float) -> float:
    """Arrival intensity for market orders."""
    return theta_g * np.exp(kappa * (float(max(liquidity, 0.0)) - lam_ref))

In [ ]:
meta = {
    'experiment': 'lobster_calibration',
    'liq_min': float(liq_min), 'liq_max': float(liq_max),
    'liq_grid': np.asarray(liq_grid),
    'theta_f': float(theta_f), 'theta_g': float(theta_g),
    'lam_ref': float(lam_ref), 'kappa': float(kappa),
    'E_marks': E_marks,
}

## 2. Stationary Distribution and Fixed-Point Computation

For given parameters $(\kappa, \theta_f, \theta_g, \lambda_r)$, the liquidity process is a continuous-time Markov chain on $\Lambda$ with generator matrix $Q$. We compute:
- the generator $Q$ from the mark space and rate functions,
- the stationary distribution $\pi$ via $\pi Q = 0$, $\sum \pi_i = 1$,
- the stationary mean $\bar{\lambda}_\pi = \sum_i \lambda_i \pi_i$,
- the fixed point $\lambda_r = \bar{\lambda}_\pi(\kappa, \lambda_r)$ via Brent's method.

In [ ]:
def idx_of_lambda(lam_val,liq_grid):
    n = liq_grid.size
    if lam_val <= liq_grid[0]:
        return 0
    if lam_val >= liq_grid[-1]:
        return n-1
    j = int(np.searchsorted(liq_grid, lam_val))
    if j == 0:
        return 0
    if j >= n:
        return n-1
    if abs(liq_grid[j] - lam_val) < abs(liq_grid[j-1] - lam_val):
        return j
    else:
        return j-1

def compute_Q(meta: Dict[str,Any], side : str) -> np.ndarray:

    local_meta = meta.copy()
    
    liq_grid = local_meta['liq_grid']
    theta_f = float(local_meta['theta_f'])
    theta_g = float(local_meta['theta_g'])
    lam_ref = float(local_meta['lam_ref'])
    kappa_f = float(local_meta.get('kappa_f', local_meta['kappa']))
    kappa_g = float(local_meta.get('kappa_g', local_meta['kappa']))

    n = liq_grid.size
    Q = np.zeros((n,n), dtype=float)
    
    for i, lam in enumerate(liq_grid):
        out_rates = np.zeros(n, dtype=float)
        for mark in local_meta['E_marks']:
            nu_mark = float(mark.get('nu', 0.0))
            if nu_mark <= 0:
                continue
            kind = mark.get('kind','')
            if kind == 'market':
                eta = float(mark.get('eta', 0.0))
                # market buy (eta>0) hits ask; market sell (eta<0) hits bid
                hits_this_side = (side == 'ask' and eta > 0) or (side == 'bid' and eta < 0)
                if not hits_this_side:
                    continue
                rate = g(lam, theta_g, kappa_g, lam_ref)
                executed = min(abs(eta), lam)   # exogenous liquidity only
                lam_new = max(0.0, lam - executed)
                j = idx_of_lambda(lam_new,liq_grid)
                out_rates[j] += nu_mark * rate

            elif kind in ('limit_buy','limit_sell'):
                affects = (side == 'ask' and kind == 'limit_sell') or (side == 'bid' and kind == 'limit_buy')
                if not affects:
                    continue
                rho = float(mark.get('rho', 0.0))
                rate = f(lam, theta_f, kappa_f, lam_ref)
                if rho >= 0:
                    lam_new = lam + rho
                else:
                    lam_new = max(0.0, lam - min(-rho, lam))
                j = idx_of_lambda(lam_new,liq_grid)
                out_rates[j] += nu_mark * rate
            else:
                continue

        # Remove self-transitions (boundary states where marks don't change state)
        out_rates[i] = 0.0
        total_out = out_rates.sum()
        if total_out > 0:
            Q[i, :] = out_rates
            Q[i, i] = -total_out
        else:
             Q[i, i] = 0.0 # absorbing (no outgoing events) -> keep zero row with diag 0
    return Q

def compute_pi(meta: Dict[str,Any], side: str, Q: Optional[np.ndarray]) -> np.ndarray:

    local_meta = meta.copy()
    
    if Q is None:
        Q = compute_Q(local_meta,side)
    n = Q.shape[0]
    A = Q.T.copy()
    b = np.zeros(n, dtype=float)
    A[-1, :] = 1.0                #set last row of A = Q transpose to 1
    b[-1] = 1.0                   #set b = (0 ... 0 1)
    try:
        pi = solve(A, b)
    except Exception:
        # fallback least squares
        x, *_ = np.linalg.lstsq(A, b, rcond=None)
        pi = x
    pi = np.maximum(pi, 0.0) #ensure positive probabilities
    # normalize 
    s = pi.sum()
    if s <= 0:
        pi = np.ones(n, dtype=float)/float(n)
    else:
        pi = pi / s

    return pi
    
def compute_stationary_mean(meta: Dict[str,Any], side : str,  Q: Optional[np.ndarray], pi: Optional[np.array]) -> float:

    local_meta = meta.copy()
    
    if pi is None:
        pi = compute_pi(meta,side,Q)
    liq_grid = local_meta['liq_grid']

    return float(np.dot(pi, liq_grid))

def compute_Q_and_pi(meta: Dict[str,Any], side : str) -> Tuple[np.ndarray, np.array, float]:
    
    local_meta = meta.copy()
    
    Q = compute_Q(local_meta, side)
    pi = compute_pi(local_meta, side, Q)
    stationary_mean = compute_stationary_mean(local_meta, side, Q, pi)
    
    return Q, pi, stationary_mean

In [ ]:
def solve_fixed_point_lambda_r(meta: Dict[str,Any], side: str='ask', tol:float=1e-8, max_iter:int=300, verbose: bool = True):

    local_meta = meta.copy()
    
    liq_grid = local_meta['liq_grid']
    liq_min = local_meta['liq_min']
    liq_max = local_meta['liq_max']

    
    def target(liq):
        local_meta['lam_ref'] = liq
        _, _, lam_pi = compute_Q_and_pi(local_meta, side)
        return lam_pi - liq

    target_min = target(liq_min)
    target_max = target(liq_max)

    if target_min * target_max < 0:
        try:
            root = brentq(target, liq_min, liq_max, xtol=tol, rtol=tol, maxiter=max_iter)
            local_meta['lam_ref'] = root
            Q, pi , stationary_mean = compute_Q_and_pi(local_meta, side)
            return root, Q, pi, stationary_mean
        except Exception as e:
            print('brentq failed')
                # fallback to fixed-point

    # damped fixed-point iteration (guaranteed existence by Brouwer in finite grid)
    x = 0.5*(liq_min + liq_max)
    local_meta['lam_ref'] = x
    alpha = 0.7
    best = {'dist': np.inf, 'lambda_r': x, 'pi': None, 'Q': None}
    for it in range(max_iter):
        Q, pi, stationary_mean = compute_Q_and_pi(local_meta, side)
        x_new = alpha * stationary_mean + (1.0-alpha) * x
        dist = abs(x_new - x)
        if dist < best['dist']:
            best.update({'dist': dist, 'lambda_r': float(x_new), 'pi': pi, 'Q': Q})
        if dist < tol:
            return float(x_new), Q, pi, stationary_mean
        x = x_new
        local_meta['lam_ref'] = x
        # adapt alpha if stalled
        if it>40 and dist > 1e-3:
            alpha = max(0.2, alpha*0.9)
        # not converged: return best approx
    print('No convergence')
    return 0,0,0,0

In [ ]:
# Verify the fixed-point solver with the initial parameters
root, Q_init, pi_init, mean_init = solve_fixed_point_lambda_r(meta)
meta['lam_ref'] = root
print(f"Fixed point: lambda_r = {root:.3f}, stationary mean = {mean_init:.3f}")

### Choice of limit-order size distribution

The limit-order size distribution $P_2$ controls the shape of the stationary liquidity measure. The distribution from BCK (`P2_vals_old`) assigns more weight to large postings, producing a **negatively skewed** stationary measure — more mass above the mean. Our distribution (`P2_vals`) assigns more weight to small postings and cancellations, producing a **slightly positively skewed**, rather **symmetric**, stationary measure that better matches the empirical observation of positively skewed best-book liquidity (see the empirical distributions in Section 3).

We should mention here that our model is fairly stiff and the range of fitting we are able to do is restrictive. This notebook and calibration procedure should mainly motivate the magnitudes for the parameter choices and not necessarily produce a perfect fit.

## 3. Empirical Analysis of LOBSTER Data

We analyse per-minute LOBSTER data for a range of tickers on 2025-07-16, between 10:30 and 15:00. For each ticker we compute level 1+2 liquidity statistics including per-minute limit-order and market-order counts, average order sizes, and time-weighted mean, standard deviation, coefficient of variation, and 95th percentile $\hat{\lambda}_{95}$ of best-book liquidity.

We want the finite model capacity $\lambda_{\max}$ (in model units) to represent a high but realistic tail level of the empirical best-book liquidity (in shares). Introducing the scaling factor
$$Q = \frac{\hat{\lambda}_{95} \text{ (shares)}}{\lambda_{\max} \text{ (model units)}},$$
one unit of model liquidity corresponds to $Q$ shares. The average absolute jump per model event is
$$\overline{\Delta\lambda} = p_{\mathrm{MO}} \cdot \mathbb{E}[|\eta|] + p_{\mathrm{LO}} \cdot \mathbb{E}[|\rho|] = 0.5 \times 1.55 + 0.5 \times 1.400 = 1.475,$$
where $p_{\mathrm{MO}} = P_3(-1) = 0.5$, $\mathbb{E}[|\eta|] = 1.55$ (from $P_1$), and $\mathbb{E}[|\rho|] = 1.400$ (from $P_2$). Denoting the empirical average in executed order size by $\widehat{\Delta\lambda}$ (shares), we choose
$$\lambda_{\max} \approx \overline{\Delta\lambda} \cdot \frac{\hat{\lambda}_{95}}{\widehat{\Delta\lambda}}.$$


In [ ]:
def analyze_lobster_comprehensive(ticker, msg_file, ob_file, interval='1min'):
    """Per-minute L1 / L1+2 liquidity statistics of one ticker (needs LOBSTER data).

    Returns (stats, df): `stats` holds one row per minute with order counts,
    average sizes, time-weighted liquidity moments, the near-mean arrival
    rates theta_f / theta_g and the derived (liq_max, Q); `df` is the
    event-level frame used by the P1 / P2 tables and the density plots.
    """
    # 1. Load Data
    msg_cols = ['time', 'type', 'order_id', 'size', 'price', 'direction']
    df_msg = pd.read_csv(msg_file, names=msg_cols, usecols=range(6))
    
    # Load Prices and Sizes for L1 and L2 (Cols 0-7)
    df_ob = pd.read_csv(ob_file, header=None, usecols=range(8))
    
    # 2. Combine Message and Book info (using default integer index to avoid duplicates)
    df = pd.concat([df_msg, df_ob], axis=1)
    df.columns = ['time', 'type', 'order_id', 'order_size', 'price', 'direction', 
                  'ap1', 'as1', 'bp1', 'bs1', 'ap2', 'as2', 'bp2', 'bs2']

    # --- ROW-LEVEL CALCULATIONS (Do this BEFORE setting the index) ---
    
    # A) Liquidity Levels
    df['liq_L1'] = (df['as1'] + df['bs1']) / 2.0
    df['liq_L12'] = (df['as1'] + df['as2'] + df['bs1'] + df['bs2']) / 2.0
    
    # B) Identify which Level the order hit
    df['is_L1'] = (df['price'] == df['ap1']) | (df['price'] == df['bp1'])
    df['is_L2'] = (df['price'] == df['ap2']) | (df['price'] == df['bp2'])
    df['is_L12'] = df['is_L1'] | df['is_L2']

    # C) MO Flow: Buy MO (Direction -1) is +, Sell MO (Direction 1) is -
    df['mo_val'] = np.where(df['type'].isin([4, 5]), df['order_size'] * (-df['direction']), 0)
    
    # D) LO Flow: (Postings - Cancellations) per side
    df['lo_val'] = 0
    # Posting (Type 1): Buy is +, Sell is -
    df.loc[df['type'] == 1, 'lo_val'] = df['order_size'] * df['direction']
    # Cancellation (Type 2/3): Remove Buy is -, Remove Sell is +
    df.loc[df['type'].isin([2, 3]), 'lo_val'] = -df['order_size'] * df['direction']

    # E) Time Duration
    df['duration'] = df['time'].shift(-1) - df['time']
    
    # --- 3. Set Time Index and Filter ---
    df['datetime'] = pd.to_datetime(df['time'], unit='s', origin='2025-07-16')
    df = df.set_index('datetime').between_time("10:30", "15:00")
    df = df.dropna(subset=['duration'])

    # --- 4. Global Activity Stats (ticker-wide, not level-specific) ---
    stats = pd.DataFrame()
    resampler = df.resample(interval)

    stats['n_post']        = df[df['type'] == 1]['type'].resample(interval).count()
    stats['n_part_cancel'] = df[df['type'] == 2]['type'].resample(interval).count()
    stats['n_full_cancel'] = df[df['type'] == 3]['type'].resample(interval).count()

    # Global size distributions (all LO / all MO regardless of level)
    lo_sizes_all = df[df['type'].isin([1, 2, 3])]['order_size']
    mo_sizes_all = df[df['type'].isin([4, 5])]['order_size']
    avg_lo_all   = lo_sizes_all.mean() if not lo_sizes_all.empty else 1
    avg_mo_all   = mo_sizes_all.mean() if not mo_sizes_all.empty else 1

    def get_bin_probs(sizes, avg):
        if sizes.empty: return 0.0, 0.0, 0.0
        p_small = (sizes <= 1.5 * avg).mean()
        p_med   = ((sizes > 1.5 * avg) & (sizes <= 2.5 * avg)).mean()
        p_large = (sizes > 2.5 * avg).mean()
        return p_small, p_med, p_large

    p_lo_s, p_lo_m, p_lo_l = get_bin_probs(lo_sizes_all, avg_lo_all)
    p_mo_s, p_mo_m, p_mo_l = get_bin_probs(mo_sizes_all, avg_mo_all)

    stats['avg_lo_size'] = lo_sizes_all.resample(interval).mean()
    stats['avg_mo_size'] = mo_sizes_all.resample(interval).mean()
    stats['p_lo_small']  = p_lo_s; stats['p_lo_med'] = p_lo_m; stats['p_lo_large'] = p_lo_l
    stats['p_mo_small']  = p_mo_s; stats['p_mo_med'] = p_mo_m; stats['p_mo_large'] = p_mo_l

    # --- 5. Per-Level Loop ---
    delta_frac = 0.05
    # E[|Δλ|] per event, weighted by event-type probabilities from P3:
    #   p_MO = P3(-1) = 0.5,  jump size ~ |e1| ~ P1  =>  E[|e1|] = 1.55
    #   p_LO = P3(1) + P3(2) = 0.5,  jump size ~ |e2| ~ |P2|  =>  E[|e2|] = 1.40
    # Both contributions use absolute sizes, consistent with avg_all_order_size in the data.
    E_delta_mo = sum(abs(k) * p for k, p in P1_vals.items())   # = 1.55
    E_delta_lo = sum(abs(k) * p for k, p in P2_vals.items())   # = 1.40
    p_mo = P3_vals.get(-1, 0)                                   # = 0.5
    p_lo = 1.0 - p_mo                                           # = 0.5
    E_delta_lambda = p_mo * E_delta_mo + p_lo * E_delta_lo      # = 0.5*1.55 + 0.5*1.40 = 1.475

    for lvl in ['L1', 'L12']:
        mask_lvl = df[f'is_{lvl}']

        # ---- 5a. Level-filtered order counts & sizes ----
        lo_sizes_lvl  = df[mask_lvl & df['type'].isin([1, 2, 3])]['order_size']
        mo_sizes_lvl  = df[mask_lvl & df['type'].isin([4, 5])]['order_size']
        all_sizes_lvl = df[mask_lvl & df['type'].isin([1, 2, 3, 4, 5])]['order_size']

        avg_lo_lvl  = lo_sizes_lvl.mean()  if not lo_sizes_lvl.empty  else 1
        avg_mo_lvl  = mo_sizes_lvl.mean()  if not mo_sizes_lvl.empty  else 1
        avg_all_lvl = all_sizes_lvl.mean() if not all_sizes_lvl.empty else 1

        stats[f'n_lo_{lvl}']  = df[mask_lvl & df['type'].isin([1, 2, 3])]['type'].resample(interval).count()
        stats[f'n_mo_{lvl}']  = df[mask_lvl & df['type'].isin([4, 5])]['type'].resample(interval).count()
        stats[f'activity_ratio_{lvl}'] = stats[f'n_lo_{lvl}'] / stats[f'n_mo_{lvl}'].replace(0, np.nan)

        stats[f'avg_lo_size_{lvl}']       = lo_sizes_lvl.resample(interval).mean()
        stats[f'avg_mo_size_{lvl}']       = mo_sizes_lvl.resample(interval).mean()
        stats[f'avg_all_order_size_{lvl}'] = all_sizes_lvl.resample(interval).mean()

        p_lo_s_lvl, p_lo_m_lvl, p_lo_l_lvl = get_bin_probs(lo_sizes_lvl, avg_lo_lvl)
        p_mo_s_lvl, p_mo_m_lvl, p_mo_l_lvl = get_bin_probs(mo_sizes_lvl, avg_mo_lvl)
        stats[f'p_lo_small_{lvl}'] = p_lo_s_lvl; stats[f'p_lo_med_{lvl}'] = p_lo_m_lvl; stats[f'p_lo_large_{lvl}'] = p_lo_l_lvl
        stats[f'p_mo_small_{lvl}'] = p_mo_s_lvl; stats[f'p_mo_med_{lvl}'] = p_mo_m_lvl; stats[f'p_mo_large_{lvl}'] = p_mo_l_lvl

        # ---- 5b. Signed flow ----
        stats[f'avg_mo_flow_{lvl}'] = df[mask_lvl]['mo_val'].resample(interval).sum()
        stats[f'avg_lo_flow_{lvl}'] = df[mask_lvl]['lo_val'].resample(interval).sum()

        # ---- 5c. Time-Weighted Liquidity ----
        w_col    = f'w_{lvl}'
        w_sq_col = f'w_sq_{lvl}'
        df[w_col]    = df[f'liq_{lvl}'] * df['duration']
        df[w_sq_col] = (df[f'liq_{lvl}']**2) * df['duration']

        total_dur = df['duration'].resample(interval).sum()
        mu    = df[w_col].resample(interval).sum() / total_dur
        sq_mu = df[w_sq_col].resample(interval).sum() / total_dur
        sigma = np.sqrt(np.maximum(0, sq_mu - mu**2))

        stats[f'mu_{lvl}']    = mu
        stats[f'sigma_{lvl}'] = sigma
        stats[f'cv_{lvl}']    = sigma / mu
        stats[f'p95_{lvl}']   = resampler[f'liq_{lvl}'].quantile(0.95)
        # struct_scale uses level-specific avg LO size
        stats[f'struct_scale_{lvl}'] = stats[f'p95_{lvl}'] / stats[f'avg_lo_size_{lvl}']

        # ---- 5d. Theta Estimation (near-mean band) ----
        row_mu     = mu.reindex(df.index, method='ffill')
        low        = row_mu * (1.0 - delta_frac)
        high       = row_mu * (1.0 + delta_frac)
        mask_close = (df[f'liq_{lvl}'] >= low) & (df[f'liq_{lvl}'] <= high)

        dur_in_band   = df[mask_close]['duration'].resample(interval).sum()
        n_lo_in_band  = df[mask_close & df['type'].isin([1, 2, 3])]['type'].resample(interval).count()
        n_mo_in_band  = df[mask_close & df['type'].isin([4, 5])]['type'].resample(interval).count()

        theta_f_emp = (n_lo_in_band / dur_in_band.replace(0, np.nan)) * 60
        theta_g_emp = (n_mo_in_band / dur_in_band.replace(0, np.nan)) * 60

        # ---- 5e. λ_max and Q Calibration ----
        # λ_max ≈ E[|Δλ|] · p95_global / avg_all_order_size_{lvl}
        # Q     = p95_global / λ_max   (shares per model unit)
        p95_global = df[f'liq_{lvl}'].quantile(0.95)
        if avg_all_lvl > 0 and p95_global > 0:
            liq_max = E_delta_lambda * p95_global / avg_all_lvl
        else:
            liq_max = 20
        Q = p95_global / float(liq_max) if p95_global > 0 else np.nan

        stats[f'liq_max_{lvl}'] = liq_max
        stats[f'Q_{lvl}']       = Q

        stats[f'theta_f_{lvl}']       = theta_f_emp
        stats[f'theta_g_{lvl}']       = theta_g_emp
        stats[f'theta_f_Q_{lvl}']     = theta_f_emp / Q
        stats[f'theta_g_Q_{lvl}']     = theta_g_emp / Q
        stats[f'lo_mg_ratio_{lvl}']   = theta_f_emp / theta_g_emp

    return stats, df


In [ ]:
# --- Needs LOBSTER data: per-ticker statistics for the seven tickers of the appendix ---
TICKERS = ['AAPL', 'AMAT', 'CSCO', 'EBAY', 'GOOG', 'INTC', 'MSFT']
FILE_SUFFIX = "2025-07-16_34140000_57660000"
DATA_PATH = "lobster_data/"

results_list = []
raw_data = {}
for ticker in TICKERS:
    msg_f = f"{DATA_PATH}{ticker}_{FILE_SUFFIX}_message_2.csv"
    book_f = f"{DATA_PATH}{ticker}_{FILE_SUFFIX}_orderbook_2.csv"
    try:
        ticker_stats, raw_data[ticker] = analyze_lobster_comprehensive(ticker, msg_f, book_f)
        median_stats = ticker_stats.median().to_frame(ticker).T
        results_list.append(median_stats)
    except Exception as e:
        print(f"Error processing {ticker}: {e}")
print('Analysing tickers done.')

final_table = pd.concat(results_list)

In [ ]:
def print_formatted_table(df, columns, title):
    print(f"\n{'='*140}")
    print(f" {title} ")
    print(f"{'='*140}")
    
    formatted_df = df[columns].copy()
    
    for col in formatted_df.columns:
        # Handle the dictionary column specifically
        if col == 'P2_vals':
            formatted_df[col] = formatted_df[col].apply(
                lambda d: {k: round(v, 3) for k, v in d.items()} if isinstance(d, dict) else d
            )
        # Handle counts
        elif col.startswith('n_'):
            formatted_df[col] = formatted_df[col].round(0).astype(int)
        # Handle all other numeric columns
        else:
            try:
                formatted_df[col] = formatted_df[col].astype(float).round(3)
            except:
                # If column isn't numeric, leave it be
                pass
            
    print(formatted_df.to_string())

### Table 1: Level 1+2 liquidity statistics for selected tickers

The table reports the empirical 95th percentile $\hat{\lambda}_{95}$, the average executed order size $\widehat{\Delta\lambda}$, the derived model capacity $\lambda_{\max}$, the scaling factor $Q$, the time-weighted mean liquidity $\hat{\lambda}$, and the empirical near-mean arrival rates $\hat{\theta}_f$ and $\hat{\theta}_g$ (per minute, within $\pm 5\%$ of $\hat{\lambda}$).

The activity ratio $\hat{\theta}_f / \hat{\theta}_g \approx 4$–$10$ motivates the order of magnitude for $\theta_f / \theta_g$ in the rate functions. However, the empirical arrival rates are too high to be used directly in numerical simulations, as orders would get triggered at almost every time step. Instead, these magnitudes motivate the *ratio* $\hat{\theta}_f / \hat{\theta}_g$, and the absolute magnitudes of $\theta_f$, $\theta_g$ are set through distributional calibration.


In [ ]:
# Table 1: Liquidity statistics for selected tickers (L1+2)
selected_tickers = ['AAPL', 'AMAT', 'CSCO', 'EBAY', 'GOOG', 'INTC', 'MSFT']
table1_cols = [
    'p95_L12', 'avg_all_order_size_L12', 'liq_max_L12', 'Q_L12',
    'mu_L12', 'theta_f_L12', 'theta_g_L12', 'lo_mg_ratio_L12'
]
print_formatted_table(
    final_table.loc[selected_tickers],
    table1_cols,
    "TABLE 1: Level 1+2 Liquidity Statistics for Selected Tickers"
)

### Table 1b: Empirical $P_2$ distribution — limit-order sizes mapped to model categories $\{-3,\ldots,3\}$

Limit-order events (LOBSTER types 1–3) are mapped to the six model size categories as follows.
Each raw order size is normalised by the ticker's average limit-order size $\bar{s}$.
The **magnitude** class is then

$$|k| = \begin{cases} 1 & s < 1.5\,\bar{s} \\ 2 & 1.5\,\bar{s} \le s < 2.5\,\bar{s} \\ 3 & s \ge 2.5\,\bar{s} \end{cases}$$

and the **sign** is positive for postings (type 1) and negative for cancellations (types 2–3).
The table reports the empirical frequency of each category together with the model $P_2$ values used in the calibration.

In [ ]:
def compute_empirical_P2(ticker_df):
    """
    Map raw LOBSTER limit-order sizes to model categories {-3,-2,-1,1,2,3}.

    Magnitude bands (relative to per-ticker average LO size s̄):
        1 : size <  1.5 · s̄   (~1× average)
        2 : 1.5 · s̄  ≤ size < 2.5 · s̄   (~2× average)
        3 : size ≥  2.5 · s̄   (~3× average)

    Sign convention:
        positive (+)  : posting       (LOBSTER type 1)
        negative (−)  : cancellation  (LOBSTER type 2 or 3)

    Returns a dict keyed by {-3,-2,-1,1,2,3} with empirical probabilities.
    """
    lo_df = ticker_df[ticker_df['type'].isin([1, 2, 3])].copy()

    if lo_df.empty:
        return {k: np.nan for k in [-3, -2, -1, 1, 2, 3]}

    avg_size = lo_df['order_size'].mean()
    if avg_size <= 0:
        return {k: np.nan for k in [-3, -2, -1, 1, 2, 3]}

    norm = lo_df['order_size'] / avg_size

    magnitude = np.where(norm < 1.5, 1,
                np.where(norm < 2.5, 2, 3)).astype(int)

    sign = np.where(lo_df['type'].values == 1, 1, -1)

    model_sizes = sign * magnitude

    total = len(model_sizes)
    counts = pd.Series(model_sizes).value_counts()

    return {k: counts.get(k, 0) / total for k in [-3, -2, -1, 1, 2, 3]}


# ── Build empirical rows ───────────────────────────────────────────────────────
ordered_keys = [-3, -2, -1, 1, 2, 3]
col_names    = [f'P2({k:+d})' for k in ordered_keys]

p2_rows = []
for t in selected_tickers:
    if t in raw_data:
        p2 = compute_empirical_P2(raw_data[t])
        p2_rows.append({'ticker': t, **p2})

df_p2_emp = pd.DataFrame(p2_rows).set_index('ticker')[ordered_keys]
df_p2_emp.columns = col_names

# ── Empirical average rows ─────────────────────────────────────────────────────
avg_row = df_p2_emp.mean(axis=0).to_frame('Emp. Average').T

outliers = ['INTC', 'CSCO']
avg_row_ex = (df_p2_emp
              .drop(index=[t for t in outliers if t in df_p2_emp.index])
              .mean(axis=0)
              .to_frame('Emp. Avg (ex INTC/CSCO)').T)

# ── Model P2 row for easy comparison ──────────────────────────────────────────
model_row = pd.DataFrame(
    {f'P2({k:+d})': [P2_vals[k]] for k in ordered_keys},
    index=['Model P2']
)

df_p2_table = pd.concat([df_p2_emp, avg_row, avg_row_ex, model_row])

# ── Print ──────────────────────────────────────────────────────────────────────
sep = '=' * 80
print(f"\n{sep}")
print("  TABLE 1b: Empirical P2 — Limit-Order Sizes in Model Categories {-3,...,3}")
print(f"{sep}")
print("  Magnitude: 1 = [0, 1.5×avg),  2 = [1.5, 2.5×avg),  3 = [2.5×avg, ∞)")
print("  Sign:  + = posting (type 1),   − = cancellation (type 2 / 3)")
print(sep)
print(df_p2_table.round(3).to_string())
print(sep)


### Table 1c: Empirical $P_1$ distribution — market-order sizes mapped to model categories $\{-3,\ldots,3\}$

Market-order events (LOBSTER types 4–5) are mapped to the six model size categories as follows.
Each raw order size is normalised by the ticker's average market-order size $\bar{s}$.
The **magnitude** class is then

$$|k| = \begin{cases} 1 & s < 1.5\,\bar{s} \\ 2 & 1.5\,\bar{s} \le s < 2.5\,\bar{s} \\ 3 & s \ge 2.5\,\bar{s} \end{cases}$$

and the **sign** is positive for buy market orders (LOBSTER direction $-1$) and negative for sell market orders (LOBSTER direction $+1$).
The table reports the empirical frequency of each category together with the model $P_1$ values used in the calibration.

In [ ]:
def compute_empirical_P1(ticker_df):
    """
    Map raw LOBSTER market-order sizes to model categories {-3,-2,-1,1,2,3}.

    Magnitude bands (relative to per-ticker average MO size s̄):
        1 : size <  1.5 · s̄   (~1× average)
        2 : 1.5 · s̄  ≤ size < 2.5 · s̄   (~2× average)
        3 : size ≥  2.5 · s̄   (~3× average)

    Sign convention (matches LOBSTER direction encoding):
        positive (+)  : buy  market order  (LOBSTER direction = -1)
        negative (−)  : sell market order  (LOBSTER direction = +1)

    Returns a dict keyed by {-3,-2,-1,1,2,3} with empirical probabilities.
    """
    mo_df = ticker_df[ticker_df['type'].isin([4, 5])].copy()

    if mo_df.empty:
        return {k: np.nan for k in [-3, -2, -1, 1, 2, 3]}

    avg_size = mo_df['order_size'].mean()
    if avg_size <= 0:
        return {k: np.nan for k in [-3, -2, -1, 1, 2, 3]}

    norm = mo_df['order_size'] / avg_size

    magnitude = np.where(norm < 1.5, 1,
                np.where(norm < 2.5, 2, 3)).astype(int)

    # direction = -1 → buy MO → positive sign; direction = +1 → sell MO → negative sign
    sign = np.where(mo_df['direction'].values == -1, 1, -1)

    model_sizes = sign * magnitude

    total = len(model_sizes)
    counts = pd.Series(model_sizes).value_counts()

    return {k: counts.get(k, 0) / total for k in [-3, -2, -1, 1, 2, 3]}


# ── Build empirical rows ───────────────────────────────────────────────────────
ordered_keys = [-3, -2, -1, 1, 2, 3]
col_names    = [f'P1({k:+d})' for k in ordered_keys]

p1_rows = []
for t in selected_tickers:
    if t in raw_data:
        p1 = compute_empirical_P1(raw_data[t])
        p1_rows.append({'ticker': t, **p1})

df_p1_emp = pd.DataFrame(p1_rows).set_index('ticker')[ordered_keys]
df_p1_emp.columns = col_names

# ── Empirical average rows ─────────────────────────────────────────────────────
avg_row = df_p1_emp.mean(axis=0).to_frame('Emp. Average').T

outliers = ['INTC', 'CSCO']
avg_row_ex = (df_p1_emp
              .drop(index=[t for t in outliers if t in df_p1_emp.index])
              .mean(axis=0)
              .to_frame('Emp. Avg (ex INTC/CSCO)').T)

# ── Model P1 row for easy comparison ──────────────────────────────────────────
model_row = pd.DataFrame(
    {f'P1({k:+d})': [P1_vals[k]] for k in ordered_keys},
    index=['Model P1']
)

df_p1_table = pd.concat([df_p1_emp, avg_row, avg_row_ex, model_row])

# ── Print ──────────────────────────────────────────────────────────────────────
sep = '=' * 80
print(f"\n{sep}")
print("  TABLE 1c: Empirical P1 — Market-Order Sizes in Model Categories {-3,...,3}")
print(f"{sep}")
print("  Magnitude: 1 = [0, 1.5×avg),  2 = [1.5, 2.5×avg),  3 = [2.5×avg, ∞)")
print("  Sign:  + = buy MO (direction −1),   − = sell MO (direction +1)")
print(sep)
print(df_p1_table.round(3).to_string())
print(sep)


### Empirical liquidity distributions

The time-weighted empirical distributions of best-book liquidity are **positively skewed** around the mean — there is more mass below the mean than above, with a right tail extending to high liquidity levels. This positive skewness motivated our choice of the limit-order size distribution $P_2$ over the BCK alternative (see Section 2).

## 4. Calibration Strategy

### Why distributional fitting over moment matching

Instead of fitting $\theta_f$ and $\theta_g$ directly to the empirical arrival rates (which far exceed what the discrete state space can handle numerically), we calibrate by minimising a distributional distance between the model's stationary measure $\pi$ and the empirical time-weighted liquidity histogram.

Two simpler approaches prove inadequate.

**Approach (A) — $\kappa$-only tuning under (E1) to match $CV_{\mathrm{model}} = CV_{\mathrm{data}}$.**
Under the self-consistency condition (E1), $\lambda_r = \lambda_\pi(\kappa, \lambda_r)$, the reference level $\lambda_r$ is not a free parameter: it is the fixed point implicitly determined by $\kappa$. Consequently, $\kappa$ alone must control both the location of the stationary mean $\lambda_\pi$ and the shape (spread) of the distribution. In practice, increasing $\kappa$ pushes the fixed point $\lambda_r(\kappa)$ toward $\Lambda_{\max}$ while monotonically decreasing $CV$: a $\kappa$ matching $CV_{\mathrm{data}}$ may exist, but the implied stationary mean $\lambda_r(\kappa)$ will generally differ from the empirical mean $\bar\lambda_{\mathrm{data}}$. For large $\kappa$, the fixed-point solver also becomes numerically unstable near the upper boundary of the state space.

**Approach (B) — joint moment matching: $\lambda_\pi(\kappa,\lambda_r) = \bar\lambda_{\mathrm{data}}$, $CV_{\mathrm{model}}(\kappa,\lambda_r) = CV_{\mathrm{data}}$.**
Freeing $\lambda_r$ as a second tuning parameter and dropping (E1) introduces internal inconsistency: $\lambda_r$ is the centering level in the rate functions $f(\lambda)=\theta_f e^{-\kappa(\lambda-\lambda_r)}$ and $g(\lambda)=\theta_g e^{\kappa(\lambda-\lambda_r)}$, so setting $\lambda_r \neq \lambda_\pi$ means these intensities are mis-centered relative to the actual stationary mean. Even ignoring consistency, two moments leave the distributional shape — skewness, tail decay, possible boundary pile-up — entirely unconstrained; and the nonlinear $2\times 2$ system need not admit a solution within the model's feasible parameter region.

One might attempt to repair Approach (A) by additionally freeing $\theta_f$ or $\theta_g$ to jointly satisfy (E1), $\lambda_\pi = \bar\lambda_{\mathrm{data}}$, and $CV_{\mathrm{model}} = CV_{\mathrm{data}}$. This is conceptually consistent but numerically fragile: the mapping from $(\kappa,\theta_f,\theta_g)$ to moments is highly nonlinear, the parameters are strongly correlated, and finite-grid boundary effects make the system ill-conditioned.

We therefore minimise the **Jensen–Shannon divergence** (JS distance) between $\pi_\kappa$ and the empirical histogram. This:

- retains (E1): the fixed point $\lambda_r(\kappa)$ is solved internally at each evaluation of the objective,
- captures the full distributional shape rather than a finite set of summary statistics,
- yields a smooth scalar objective that naturally accommodates additional free parameters ($\theta_f$, $\theta_g$) without overfitting to isolated moments.

In [ ]:
def distance_metric(pi_model: np.ndarray,
                              pi_emp: np.ndarray,
                              metric: str = 'JS',
                              eps: float = 1e-12,
                              grid: np.ndarray = None) -> float:
        """
        Compute a distance/divergence between two discrete distributions pi_model and pi_emp.
        - pi_model, pi_emp: arrays of same length, nonnegative (need not sum to 1).
        - metric: one of 'KL', 'symKL', 'JS', 'EMD', 'L1', 'L2'
        - eps: small additive smoothing for KL/JS to avoid zeros.
        - grid: optional array of grid locations (same length). If provided, EMD uses spacing in grid;
                otherwise assumes unit spacing.
        Returns a nonnegative scalar. For JS and KL, returns the natural-log base nats value (not bits).
        Notes:
          - For KL and symKL we apply additive smoothing (Laplace) then renormalize.
          - For JS we also smooth before computing.
          - For EMD we compute 1D Wasserstein using CDF difference (faster than sampling).
        """
        # convert to floats and ensure 1D
        p = np.asarray(pi_model, dtype=float).ravel().copy()
        q = np.asarray(pi_emp, dtype=float).ravel().copy()
        if p.shape != q.shape:
            raise ValueError("pi_model and pi_emp must have same shape")
    
        # if both zero vectors, distance = 0
        if np.all(p == 0) and np.all(q == 0):
            return 0.0
    
        # normalize (but keep possibility of zeros)
        sp = float(np.sum(p))
        sq = float(np.sum(q))
        if sp <= 0:
            p_norm = np.full_like(p, 1.0 / p.size)
        else:
            p_norm = p / sp
        if sq <= 0:
            q_norm = np.full_like(q, 1.0 / q.size)
        else:
            q_norm = q / sq
    
        metric = metric.upper()
    
        def kl_div(a, b):
            """KL divergence D(a||b) in nats, assumes a,b are probability vectors and b>0 everywhere"""
            mask = a > 0.0
            # safe log: only where a>0; assume b>0
            return float(np.sum(a[mask] * np.log(a[mask] / b[mask])))
    
        if metric == 'KL':
            # Additive smoothing then renormalize
            a = p_norm + eps
            b = q_norm + eps
            a /= a.sum()
            b /= b.sum()
            return kl_div(a, b)
    
        if metric == 'SYMKL' or metric == 'SYMMETRIC_KL' or metric == 'SYMMETRIC-KL'.upper():
            a = p_norm + eps
            b = q_norm + eps
            a /= a.sum()
            b /= b.sum()
            return 0.5 * (kl_div(a, b) + kl_div(b, a))
    
        if metric == 'JS' or metric == 'JSD' or metric == 'JENSEN-SHANNON':
            a = p_norm + eps
            b = q_norm + eps
            a /= a.sum()
            b /= b.sum()
            m = 0.5 * (a + b)
            return 0.5 * kl_div(a, m) + 0.5 * kl_div(b, m)
    
        if metric == 'L1' or metric == 'TV' or metric == 'TOTAL_VARIATION':
            return float(0.5 * np.sum(np.abs(p_norm - q_norm))) if metric == 'TV' else float(np.sum(np.abs(p_norm - q_norm)))
    
        if metric == 'L2':
            return float(np.sqrt(np.sum((p_norm - q_norm)**2)))
    
        if metric == 'EMD' or metric == 'WASSERSTEIN' or metric == 'W1':
            # 1D EMD between discrete distributions on ordered grid.
            # If grid provided, use it; otherwise assume uniform unit spacing
            if grid is None:
                # discrete 1D EMD = sum |CDF_p - CDF_q| * dx where dx=1 (unit spacing)
                cdf_p = np.cumsum(p_norm)
                cdf_q = np.cumsum(q_norm)
                emd = float(np.sum(np.abs(cdf_p - cdf_q)))
                # If you want true Wasserstein-1 over spacing, multiply by dx (units)
                return emd
            else:
                # If grid provided, compute EMD as integral |CDF diff| * dx
                grid = np.asarray(grid, dtype=float)
                if grid.size != p_norm.size:
                    raise ValueError("grid must have same length as distributions")
                dx = np.diff(grid)
                if np.any(dx <= 0):
                    raise ValueError("grid must be strictly increasing")
                # compute CDF at bin edges: use left Riemann integration: sum |cdf_diff[:-1]| * dx
                cdf_p = np.cumsum(p_norm)
                cdf_q = np.cumsum(q_norm)
                # use dx for weights; here dx aligns with bins i->i+1
                emd = float(np.sum(np.abs(cdf_p[:-1] - cdf_q[:-1]) * dx))
                return emd
    
        # fallback
        raise ValueError(f"Unknown metric '{metric}'")

In [ ]:
def calibrate_model_comprehensive(
    ticker: str,
    ob_file: str,
    msg_file: str,
    meta: Dict[str, Any],
    params_to_tune: List[str] = ['kappa'],
    side: str = 'both',
    metric: str = 'JS',
    time_weighted: bool = True,
    liquidity_level: str = 'L1',
    auto_liq_max: bool = False,  # True: derive liq_max from data (ceil to ×5), overrides meta['liq_max']
    verbose: bool = True
) -> Dict[str, Any]:

    # --- 1. Empirical Data Preparation ---
    ob_cols = [1, 3, 5, 7] if liquidity_level == 'L12' else [1, 3]
    df_ob  = pd.read_csv(ob_file,  header=None, usecols=ob_cols)
    df_msg = pd.read_csv(msg_file, names=['time', 'type', 'id', 'size', 'price', 'dir'], usecols=range(6))

    df = pd.concat([df_msg[['time', 'type']], df_ob], axis=1)

    if liquidity_level == 'L12':
        df['ask_depth'] = df.iloc[:, 2] + df.iloc[:, 4]
        df['bid_depth'] = df.iloc[:, 3] + df.iloc[:, 5]
    else:
        df['ask_depth'] = df.iloc[:, 2]
        df['bid_depth'] = df.iloc[:, 3]

    df['datetime'] = pd.to_datetime(df['time'], unit='s', origin='2025-07-16')
    df = df.set_index('datetime').between_time("10:30", "15:00")

    df['weight'] = (df['time'].shift(-1) - df['time']) if time_weighted else 1.0
    df = df.dropna(subset=['weight'])

    # Empirical liquidity series.
    # For side='both' use (ask+bid)/2, matching liq_L1 = (as1+bs1)/2 in
    # analyze_lobster_comprehensive so that p95, Q, and emp_pi are consistent.
    if side == 'both':
        vals = ((df['ask_depth'] + df['bid_depth']) / 2.0).values.astype(float)
    elif side == 'ask':
        vals = df['ask_depth'].values.astype(float)
    else:
        vals = df['bid_depth'].values.astype(float)
    weights = df['weight'].values.astype(float)

    emp_mean_shares = np.sum(vals * weights) / np.sum(weights)
    p95_shares      = np.quantile(vals, 0.95)

    # --- 2. Fix liq_max and Q ---
    # When auto_liq_max=True, both liq_max AND Q are derived from the same p95_shares
    # computed just above, so Q = p95_shares / liq_max is fully self-consistent.
    # E[|Δλ|] per event, weighted by event-type probabilities from P3:
    #   p_MO = P3(-1) = 0.5,  jump size ~ |e1| ~ P1  =>  E[|e1|] = 1.55
    #   p_LO = P3(1) + P3(2) = 0.5,  jump size ~ |e2| ~ |P2|  =>  E[|e2|] = 1.40
    _E_DELTA_MO  = sum(abs(k) * p for k, p in P1_vals.items())   # = 1.55
    _E_DELTA_LO  = sum(abs(k) * p for k, p in P2_vals.items())   # = 1.40
    _p_mo        = P3_vals.get(-1, 0)                             # = 0.5
    _p_lo        = 1.0 - _p_mo                                    # = 0.5
    _E_DELTA_LAM = _p_mo * _E_DELTA_MO + _p_lo * _E_DELTA_LO     # = 0.5*1.55 + 0.5*1.40 = 1.475
    if auto_liq_max:
        # Compute avg executed order size from the same time window as the ob data
        df_msg['_dt'] = pd.to_datetime(df_msg['time'], unit='s', origin='2025-07-16')
        df_msg_flt    = df_msg.set_index('_dt').between_time("10:30", "15:00")
        _ord_mask     = df_msg_flt['type'].isin([1, 2, 3, 4, 5])
        _avg_ord      = df_msg_flt.loc[_ord_mask, 'size'].mean() if _ord_mask.any() else 1.0
        _raw_lmax     = (_E_DELTA_LAM * p95_shares / _avg_ord) if _avg_ord > 0 \
                        else float(meta.get('liq_max', 40))
        l_max             = int(math.ceil(_raw_lmax / 5.0) * 5)
        raw_liq_max_auto  = _raw_lmax
    else:
        l_max            = int(meta.get('liq_max', 40))
        raw_liq_max_auto = None

    Q = p95_shares / l_max

    # Pre-calculate empirical distribution once.
    # Q and l_max are fixed, so this target never changes during optimisation.
    emp_pi = np.zeros(l_max + 1)
    inds   = np.clip(np.round(vals / Q), 0, l_max).astype(int)
    for iv, w in zip(inds, weights):
        emp_pi[iv] += w
    emp_pi /= emp_pi.sum()

    # --- 3. Optimization Setup ---
    jump_keys   = [-3, -2, -1, 1, 2, 3]
    # Default P2 distribution matches the module-level P2_vals (used when tuning_probs=False)
    default_p_lo = dict(P2_vals)

    # liq_max is always fixed (never tuned)
    active_params = [p for p in params_to_tune if p not in ['liq_max', 'P2_probs']]
    tuning_probs  = 'P2_probs' in params_to_tune

    initial_guess = []
    bounds        = []
    bounds_map = {
        'kappa':   (0.001, 1.0),
        'kappa_f': (0.001, 2.0),
        'kappa_g': (0.001, 2.0),
        'theta_f': (0.1,  100.0),
        'theta_g': (0.05,  95.0)
    }

    for p in active_params:
        initial_guess.append(meta.get(p, (bounds_map[p][0] + bounds_map[p][1]) / 2))
        bounds.append(bounds_map[p])

    if tuning_probs:
        initial_guess.extend([default_p_lo[k] for k in jump_keys])
        bounds.extend([(0.001, 0.8)] * 6)

    # --- 4. Objective Function ---
    def objective(x):
        m_curr = meta.copy()
        m_curr['liq_max']  = l_max
        m_curr['liq_grid'] = np.arange(l_max + 1)

        idx = 0
        for p in active_params:
            m_curr[p] = x[idx]; idx += 1

        # Stability: θ_f must be strictly above θ_g
        if m_curr.get('theta_f', 10) <= m_curr.get('theta_g', 5):
            return 1e6

        p_lo_curr = default_p_lo.copy()
        if tuning_probs:
            p_vals      = x[idx: idx + 6]
            p_vals_norm = p_vals / np.sum(p_vals)
            p_lo_curr   = {k: v for k, v in zip(jump_keys, p_vals_norm)}
            m_curr['E_marks'] = build_E_marks(P1_vals, p_lo_curr, P3_vals)

        try:
            sides_calc = ['ask', 'bid'] if side == 'both' else [side]
            pis        = [solve_fixed_point_lambda_r(m_curr, s)[2] for s in sides_calc]
            pi_model   = np.mean(np.vstack(pis), axis=0)
            return distance_metric(pi_model, emp_pi, metric=metric)
        except:
            return 1e6

    # --- 5. Constraints (Jump Probabilities) ---
    constraints = []
    if tuning_probs:
        p_start = len(active_params)
        constraints.append({'type': 'eq',   'fun': lambda x: np.sum(x[p_start:]) - 1.0})
        constraints.append({'type': 'ineq', 'fun': lambda x: x[p_start + 1] - x[p_start]})
        constraints.append({'type': 'ineq', 'fun': lambda x: x[p_start + 2] - x[p_start + 1]})
        constraints.append({'type': 'ineq', 'fun': lambda x: x[p_start + 3] - x[p_start + 4]})
        constraints.append({'type': 'ineq', 'fun': lambda x: x[p_start + 4] - x[p_start + 5]})

    # --- 6. Run Optimization ---
    mode_str = "Time-Weighted" if time_weighted else "Event-Weighted"
    if verbose:
        print(f"== Fixed-Target Calibration: {ticker} | λ_max: {l_max} | Q: {Q:.3f} | {mode_str} ==")

    res = minimize(objective, initial_guess, bounds=bounds, constraints=constraints, method='SLSQP')

    # --- 7. Results Extraction ---
    best_meta              = copy.deepcopy(meta)
    best_meta['liq_max']   = l_max
    best_meta['liq_grid']  = np.arange(l_max + 1)

    idx = 0
    for p in active_params:
        best_meta[p] = float(res.x[idx]); idx += 1

    best_p_lo = default_p_lo.copy()
    if tuning_probs:
        best_p_vals      = res.x[idx: idx + 6]
        best_p_vals_norm = best_p_vals / np.sum(best_p_vals)
        best_p_lo        = {k: v for k, v in zip(jump_keys, best_p_vals_norm)}
        best_meta['E_marks'] = build_E_marks(P1_vals, best_p_lo, P3_vals)

    # Final stationary distribution
    _, _, pi_final, lam_pi_units = solve_fixed_point_lambda_r(best_meta, 'ask' if side == 'both' else side)
    best_meta['lam_ref'] = float(lam_pi_units)

    if verbose:
        print(f"Calibration Complete. Score ({metric}): {res.fun:.6f}")
        kf_str = f", κ_f: {best_meta['kappa_f']:.4f}" if 'kappa_f' in best_meta else ""
        kg_str = f", κ_g: {best_meta['kappa_g']:.4f}" if 'kappa_g' in best_meta else ""
        print(f"Kappa: {best_meta['kappa']:.4f}{kf_str}{kg_str}, θ_f: {best_meta['theta_f']:.2f}, θ_g: {best_meta['theta_g']:.2f}")

    return {
        'ticker':            ticker,
        'liquidity_level':   liquidity_level,
        'mode_str':          mode_str,
        'meta_out':          best_meta,
        'metric':            metric,
        'dist':              res.fun,
        'Q':                 Q,
        'raw_liq_max':       raw_liq_max_auto,  # float before ×5 ceiling; None if auto_liq_max=False
        'p_lo':              best_p_lo,
        'pi_model':          pi_final,
        'pi_emp':            emp_pi,
        'emp_mean_shares':   emp_mean_shares,
        'model_mean_shares': lam_pi_units * Q,
        'scalar_params':     active_params,
        'tuning_probs':      tuning_probs,
        'jump_keys':         jump_keys,
    }


## 5. Calibration Results

We report representative calibration outputs obtained from initial values $(\theta_f, \theta_g) = (100, 20)$ and from data-driven $\lambda_{\max}$ as derived in Table 1, rounded up to the next multiple of five. The optimiser tunes $(\kappa, \theta_f, \theta_g)$ while enforcing the fixed-point condition $\lambda_r = \bar{\lambda}_\pi$ inside each objective evaluation.

In [ ]:
# --- Needs LOBSTER data ---
TICKERS       = ['AAPL', 'AMAT', 'CSCO', 'EBAY', 'GOOG', 'INTC', 'MSFT']
FILE_SUFFIX   = "2025-07-16_34140000_57660000"
DATA_PATH     = "lobster_data/"
metric        = 'JS'
time_weighted = True

calib_results_dyn = {}
calib_summary_dyn = []

print("== Calibration: data-driven lambda_max (ceil(raw/5)*5), L1+2 ==\n")

for ticker in TICKERS:
    msg_f  = f"{DATA_PATH}{ticker}_{FILE_SUFFIX}_message_2.csv"
    book_f = f"{DATA_PATH}{ticker}_{FILE_SUFFIX}_orderbook_2.csv"

    try:
        emp_stats = final_table.loc[ticker]

        meta_ticker            = meta.copy()
        meta_ticker['theta_f'] = 100
        meta_ticker['theta_g'] = 20

        t0  = time.time()
        res = calibrate_model_comprehensive(
            ticker=ticker,
            ob_file=book_f,
            msg_file=msg_f,
            meta=meta_ticker,
            params_to_tune=['kappa', 'theta_f', 'theta_g'], # 
            side='both',
            metric=metric,
            time_weighted=time_weighted,
            liquidity_level='L12',
            auto_liq_max=True,
            verbose=False,
        )
        elapsed = time.time() - t0
        calib_results_dyn[f"{ticker}_L12"] = res

        m_out = res['meta_out']
        grid  = m_out['liq_grid']
        pi    = res['pi_model']
        Q     = res['Q']

        m_mu  = float(np.sum(grid * pi))
        m_var = float(np.sum((grid ** 2) * pi)) - m_mu ** 2
        m_cv  = float(np.sqrt(max(0.0, m_var)) / m_mu) if m_mu > 0 else 0.0

        e_mu_model = emp_stats['mu_L12'] / Q
        e_cv       = emp_stats['cv_L12']

        calib_summary_dyn.append({
            'ticker':      ticker,
            'js_dist':     res['dist'],
            'liq_max':     m_out['liq_max'],
            'kappa':       m_out['kappa'],
            'theta_f':     m_out['theta_f'],
            'theta_g':     m_out['theta_g'],
            'model_mean':  m_mu,
            'emp_mean':    e_mu_model,
            'model_cv':    m_cv,
            'emp_cv':      e_cv,
        })
        print(f"  {ticker:6s} | lam_max={m_out['liq_max']:3d} "
              f"| Q={Q:8.3f} | JS={res['dist']:.5f} | {elapsed:.1f}s")

    except Exception as e:
        print(f"  ERROR {ticker}: {e}")

df_calib_dyn_L12 = pd.DataFrame(calib_summary_dyn).set_index('ticker')

### Table 2: Calibration output

The table lists the fitted resilience parameter $\kappa$, arrival parameters $\theta_f$ and $\theta_g$, the JS distance, the stationary mean of the model $\bar{\lambda}_\pi$ and the empirical mean converted to model units $\hat{\lambda}/Q$, and the coefficient of variation for the model ($CV_\pi$) and empirical liquidity ($CV_{\mathrm{emp}}$).

The JS objective primarily captures the shape of the stationary distribution. In the calibration results, we observe parameter correlations and a relatively flat objective along directions that scale both $\theta_f$ and $\theta_g$.

In [ ]:
report_cols = ['js_dist', 'liq_max', 'kappa', 'theta_f', 'theta_g',
               'model_mean', 'emp_mean', 'model_cv', 'emp_cv']
print_formatted_table(
    df_calib_dyn_L12,
    report_cols,
    "TABLE 2: Calibration Output (JS | Time-Weighted | L1+2)"
)

## 6. Benchmark Parameter Selection

The calibration does not aim at a universal fit across all assets but at parameter magnitudes representative of the core liquid-ticker regime in which the model is applied. JS distances around $0.02$--$0.03$ indicate close distributional agreement while preserving a single parsimonious parameter set.

From Table 2 (L1+2, time-weighted JS), the core tickers AAPL, AMAT, EBAY, GOOG and MSFT reach JS distances in $[0.018, 0.033]$. CSCO and INTC are structural outliers: their books are anomalously deep ($\lambda_{\max} \in \{35, 60\}$), their resilience minimal ($\kappa \in \{0.039, 0.017\}$) and their empirical coefficient of variation ($0.160$, $0.115$) far narrower than the model's. They are excluded from the benchmark anchor.

**Activity parameters.** The stationary law is invariant under proportional rescalings $(\theta_f, \theta_g) \mapsto (c\,\theta_f, c\,\theta_g)$, so the JS objective depends on $(\theta_f, \theta_g)$ only through the ratio $\theta_f/\theta_g$; the optimiser's $\theta_f \approx 100$ is the upper bound along this flat direction. The informative quantity is the ratio $\theta_f/\theta_g \approx 4$, and we round to $\theta_f = 100$, $\theta_g = 25$.

**Resilience.** $\kappa$ controls the concentration of the stationary law. Across the core group it lies in $[0.070, 0.119]$; we take $\kappa = 0.1$.

**State-space capacity.** The core $\lambda_{\max}$ values are 15 (AMAT, MSFT), 20 (AAPL, EBAY) and 25 (GOOG); $\lambda_{\max} = 20$ is the central value and keeps the state space of the QVI problems small.

**Reference liquidity.** With these choices the fixed-point condition $\lambda_r = \bar{\lambda}_\pi(\kappa, \lambda_r)$ determines $\lambda_r$.

In summary (thesis eq. (benchmark_parameters)):
$$
\theta_f = 100\,,\qquad \theta_g = 25\,,\qquad \kappa = 0.1\,,\qquad \lambda_{\max} = 20\,,\qquad \lambda_r = 12.4812\,.
$$

In [ ]:
# =====================================================================
# Benchmark parameters of the thesis, eq. (benchmark_parameters); lambda_r from the fixed point
# =====================================================================
theta_f_bm = 100.0
theta_g_bm = 25.0
kappa_bm   = 0.1
liq_max_bm = 20
liq_grid_bm = np.arange(0, liq_max_bm + 1, 1.0)
E_marks     = build_E_marks(P1_vals, P2_vals, P3_vals)

meta_verify = meta.copy()
meta_verify['theta_f']  = theta_f_bm
meta_verify['theta_g']  = theta_g_bm
meta_verify['kappa']    = kappa_bm
meta_verify['liq_max']  = liq_max_bm
meta_verify['liq_min']  = 0
meta_verify['liq_grid'] = liq_grid_bm
meta_verify['lam_ref']  = 10.0  # starting value for fixed-point solve
meta_verify['E_marks']  = E_marks

_, _, _, lam_ref_bm = solve_fixed_point_lambda_r(meta_verify)

liq_grid_bm = np.arange(0, liq_max_bm + 1, 1.0)

print("Benchmark parameters loaded.")
print(f"  theta_f={theta_f_bm}, theta_g={theta_g_bm}, kappa={kappa_bm}, "
      f"lam_max={liq_max_bm}, lam_ref={lam_ref_bm:.4f}")

# 7. Volume Imbalance Extension: Estimation of $\hat{\kappa}$

The resilience model is extended by a dependence on the volume imbalance
$$\lambda^{\mathrm{VI}} = \frac{\lambda^{\mathrm{bid}} - \lambda^{\mathrm{ask}}}{\lambda^{\mathrm{bid}} + \lambda^{\mathrm{ask}}} \in [-1,1]$$
through the rate functions
$$f^+(\lambda) = \theta_f e^{-\kappa(\lambda - \lambda_r)}, \quad f^-(\lambda, \lambda^{\mathrm{VI}}) = \theta_f e^{-\kappa(\lambda - \lambda_r) + \hat{\kappa}\lambda^{\mathrm{VI}}}, \quad g(\lambda, \lambda^{\mathrm{VI}}) = \theta_g e^{\kappa(\lambda - \lambda_r) + \hat{\kappa}\lambda^{\mathrm{VI}}}$$
for postings, cancellations and market orders respectively; at $\hat{\kappa} = 0$ the resilience model is recovered. The benchmark parameters $\theta_f = 100$, $\theta_g = 25$, $\kappa = 0.1$, $\lambda_{\max} = 20$, $\lambda_r \approx 12.481$ of Section 6 are kept fixed.

### Conditional maximum likelihood

For each LOBSTER event at time $t_i$ we record the **pre-event** book $(\lambda^{\mathrm{bid}}_{t_i-}, \lambda^{\mathrm{ask}}_{t_i-})$ (in model units, scaled by $Q = \hat{\lambda}_{95}/\lambda_{\max}$) and its imbalance $\lambda^{\mathrm{VI}}_{t_i-}$, and classify the event into one of six classes: posting, cancellation (each per side) and market order (per direction). Their intensities are
$$
\mu^{\mathrm{LO}^+,\,\mathrm{side}}_i = \nu^{+} f^{+}(\lambda^{\mathrm{side}}_{t_i-}),\qquad
\mu^{\mathrm{LO}^-,\,\mathrm{side}}_i(\hat\kappa) = \nu^{-} f^{-}(\lambda^{\mathrm{side}}_{t_i-}, \pm\lambda^{\mathrm{VI}}_{t_i-}),\qquad
\mu^{\mathrm{MO}}_i(\hat\kappa) = \nu^{\mathrm{mo}}\, g(\lambda^{\mathrm{side}}_{t_i-}, \pm\lambda^{\mathrm{VI}}_{t_i-}),
$$
with the sign $+$ for the bid side (cancellations) and for buy market orders, $-$ for the ask side and sell market orders, and mark-space weights $\nu^{+} = \tfrac14\sum_{k>0}P^2(k)$, $\nu^{-} = \tfrac14\sum_{k<0}P^2(k)$, $\nu^{\mathrm{mo}} = \tfrac12\sum_{k>0}P^1(k)$. Restricting to informative events $\mathcal{I}_\varepsilon = \{i : |\lambda^{\mathrm{VI}}_{t_i-}| \ge \varepsilon\}$, the estimate is
$$
\hat\kappa^\ast = \arg\max_{\hat\kappa} \sum_{i \in \mathcal{I}_\varepsilon} \log \frac{\mu_i^{k_i}(\hat\kappa)}{\sum_{j} \mu_i^{j}(\hat\kappa)}\,,
$$
the conditional log-likelihood of the observed event classes $k_i$ given that an event occurred. Postings do not depend on $\hat{\kappa}$ but enter both as observations and in the normalisation.

In [ ]:
# Conditional MLE for hat_kappa (pre-event book states, all six event classes).
# Needs LOBSTER data.

NU_POST   = 0.25 * sum(v for k, v in P2_vals.items() if k > 0)   # nu^+
NU_CANCEL = 0.25 * sum(v for k, v in P2_vals.items() if k < 0)   # nu^-
NU_MO     = 0.50 * sum(v for k, v in P1_vals.items() if k > 0)   # nu^mo

T_START, T_END = 10.5 * 3600, 15.0 * 3600   # 10:30 -- 15:00

# event classes
ASK_CANCEL, BID_CANCEL, BUY_MO, SELL_MO, BID_POST, ASK_POST = range(6)


def load_lobster(msg_file, ob_file):
    """Time, type, direction and the L1+2 liquidity per side (in shares)."""
    msg = pd.read_csv(msg_file, header=None, usecols=range(6),
                      names=['time', 'type', 'order_id', 'size', 'price', 'direction'])
    ob = pd.read_csv(ob_file, header=None, usecols=[1, 3, 5, 7], names=['as1', 'bs1', 'as2', 'bs2'])
    lam_ask = (ob['as1'] + ob['as2']).to_numpy(float)
    lam_bid = (ob['bs1'] + ob['bs2']).to_numpy(float)
    n = min(len(msg), len(lam_ask))
    return (msg['time'].to_numpy(float)[:n], msg['type'].to_numpy(int)[:n],
            msg['direction'].to_numpy(int)[:n], lam_bid[:n], lam_ask[:n])


def classify_events(types, dirs):
    """LOBSTER (type, direction) -> event class (-1 = not used).

    Type 1: posting; 2, 3: cancellation; 4, 5: execution (market order).
    For cancellations `direction` is the side of the cancelled order; for
    executions it is the side of the RESTING order, so direction -1 (a sell
    limit order executed) means a buy market order hit the ask.
    """
    ec = np.full(len(types), -1, dtype=int)
    cancel = np.isin(types, [2, 3])
    ec[cancel & (dirs == -1)] = ASK_CANCEL
    ec[cancel & (dirs == 1)] = BID_CANCEL
    mo = np.isin(types, [4, 5])
    ec[mo & (dirs == -1)] = BUY_MO
    ec[mo & (dirs == 1)] = SELL_MO
    post = types == 1
    ec[post & (dirs == 1)] = BID_POST
    ec[post & (dirs == -1)] = ASK_POST
    return ec


def estimate_hat_kappa(msg_file, ob_file, epsilon=0.05, lam_ref=None):
    """Conditional MLE of hat_kappa for one ticker at imbalance threshold epsilon.

    Book states are taken PRE-event (LOBSTER row i-1 is the book before message i).
    Liquidity is scaled to model units by Q = p95 / liq_max_bm with the p95 of the
    side-averaged L1+2 liquidity over the session. Returns the estimate, a
    curvature-based standard error and the event counts.
    """
    if lam_ref is None:
        lam_ref = lam_ref_bm
    time_a, types, dirs, lam_bid, lam_ask = load_lobster(msg_file, ob_file)

    in_session = (time_a >= T_START) & (time_a <= T_END)
    p95 = np.nanquantile((lam_bid[in_session] + lam_ask[in_session]) / 2, 0.95)
    q_scale = p95 / liq_max_bm

    ec_all = classify_events(types, dirs)
    rows = np.where(in_session & (ec_all >= 0))[0]
    sidx = rows - 1                      # pre-event book state
    keep = sidx >= 0
    rows, sidx = rows[keep], sidx[keep]

    tot = lam_bid[sidx] + lam_ask[sidx]
    vi = np.divide(lam_bid[sidx] - lam_ask[sidx], tot, out=np.zeros_like(tot), where=tot > 0)
    sel = np.abs(vi) >= epsilon
    vi, sidx, ec = vi[sel], sidx[sel], ec_all[rows[sel]]

    lam_b = lam_bid[sidx] / q_scale
    lam_a = lam_ask[sidx] / q_scale
    f_bid = f_rate(lam_b, lam_ref=lam_ref)
    f_ask = f_rate(lam_a, lam_ref=lam_ref)
    g_bid = g_rate(lam_b, lam_ref=lam_ref)
    g_ask = g_rate(lam_a, lam_ref=lam_ref)

    def loglik(hk):
        ep, en = np.exp(hk * vi), np.exp(-hk * vi)
        rates = {ASK_POST:   f_ask * NU_POST,
                 BID_POST:   f_bid * NU_POST,
                 ASK_CANCEL: f_ask * en * NU_CANCEL,
                 BID_CANCEL: f_bid * ep * NU_CANCEL,
                 BUY_MO:     g_ask * ep * NU_MO,
                 SELL_MO:    g_bid * en * NU_MO}
        total = sum(rates.values())
        observed = np.select([ec == k for k in rates], [rates[k] for k in rates])
        return float(np.sum(np.log(observed / total + 1e-300)))

    res = minimize_scalar(lambda h: -loglik(h), bounds=(-1.0, 3.0), method='bounded',
                          options={'xatol': 1e-8})
    hk = float(res.x)
    h = 1e-3
    curv = (loglik(hk + h) - 2 * loglik(hk) + loglik(hk - h)) / h**2
    se = float(1.0 / math.sqrt(-curv)) if curv < 0 else float('nan')

    return {'hat_kappa': hk, 'se': se, 'n_events': int(len(ec)),
            'n_postings': int(np.sum((ec == BID_POST) | (ec == ASK_POST))),
            'Q': float(q_scale), 'epsilon': epsilon}

In [ ]:
# --- Needs LOBSTER data: Table hat_kappa of the thesis ---
TICKERS_FIT = ['AAPL', 'AMAT', 'CSCO', 'EBAY', 'GOOG', 'INTC', 'MSFT']
CORE_TICKERS = ['AAPL', 'AMAT', 'EBAY', 'GOOG', 'MSFT']
FILE_SUFFIX = "2025-07-16_34140000_57660000"
DATA_PATH = "lobster_data/"
epsilons = [0.02, 0.05, 0.10]

mle_results = {}
for ticker in TICKERS_FIT:
    msg_f  = f"{DATA_PATH}{ticker}_{FILE_SUFFIX}_message_2.csv"
    book_f = f"{DATA_PATH}{ticker}_{FILE_SUFFIX}_orderbook_2.csv"
    for eps in epsilons:
        mle_results[(ticker, eps)] = estimate_hat_kappa(msg_f, book_f, epsilon=eps)
    print(f"  {ticker} done")

hk_table = pd.DataFrame(
    {f"eps={eps:.2f}": [mle_results[(t, eps)]['hat_kappa'] for t in TICKERS_FIT] for eps in epsilons},
    index=TICKERS_FIT,
)
hk_table['range'] = hk_table.max(axis=1) - hk_table.min(axis=1)
hk_table['se(eps=0.05)'] = [mle_results[(t, 0.05)]['se'] for t in TICKERS_FIT]
print(hk_table.round(4))

core_median = float(hk_table.loc[CORE_TICKERS, 'eps=0.05'].median())
print(f"\nCore-group median at eps = 0.05: {core_median:.4f}")

### Benchmark choice of $\hat{\kappa}$

The estimates are stable in $\varepsilon$ (the maximal variation across thresholds is below $0.005$ for every ticker). The core-group values span $[1.191, 1.684]$ with median $1.469$; the outliers CSCO and INTC show the highest imbalance sensitivities. Based on the core-group median the thesis sets
$$\hat{\kappa} = 1.5$$
as the empirical benchmark. The volume-imbalance tilt leaves the calibrated marginal liquidity law essentially unchanged: over $\hat{\kappa} \in [0, 2]$ the JS distance between the ask-side marginal and its $\hat{\kappa} = 0$ baseline stays below $1.1 \cdot 10^{-3}$, one to two orders of magnitude below the calibration residuals of Table 2.